# 🤖 Notebook 04 — Modelling & Evaluation

**Purpose:** Train 10 candidate models on the engineered feature matrix, evaluate on the held-out test set, select the best model, and interpret results with SHAP.

**Models evaluated:**
1. Logistic Regression (baseline)
2. Decision Tree
3. Random Forest
4. Gradient Boosting
5. XGBoost (primary model)
6. Linear SVC (calibrated)
7. Naive Bayes
8. Neural Network (MLP)
9. Voting Ensemble (LR + RF + XGB)
10. Stacking Ensemble

**Models excluded from v2 pipeline:**
- KNN — see note below

**Run order:** 01 → 02 → 03b → 03a → **04** → 05

In [ ]:
# ============================================================
# QUICK TEST MODE
# Set to True to run on a small subset (fast sanity check ~2 min).
# Set to False for the full production run.
# ============================================================
QUICK_TEST      = True   # <-- CHANGE TO False FOR FINAL RUN
QUICK_TEST_ROWS = 5_000

## Imports

In [ ]:
import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    VotingClassifier, StackingClassifier
)
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier

from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score, recall_score,
    confusion_matrix, ConfusionMatrixDisplay, roc_curve,
    average_precision_score
)

try:
    import shap
    HAS_SHAP = True
except ImportError:
    HAS_SHAP = False
    print("[WARN] shap not installed — SHAP cell will be skipped. Install with: pip install shap")

warnings.filterwarnings("ignore")
print("✓ Imports complete")

## Load Data & Apply Quick-Test Mode

In [ ]:
OUTPUTS_PATH = "data"
FIGURES_PATH = os.path.join(OUTPUTS_PATH, "figures")
RESULTS_PATH = os.path.join(OUTPUTS_PATH, "results")
os.makedirs(FIGURES_PATH, exist_ok=True)
os.makedirs(RESULTS_PATH, exist_ok=True)

# Load from notebook 03a outputs
X_train = pd.read_parquet(os.path.join(OUTPUTS_PATH, "X_train.parquet"))
X_val   = pd.read_parquet(os.path.join(OUTPUTS_PATH, "X_val.parquet"))
X_test  = pd.read_parquet(os.path.join(OUTPUTS_PATH, "X_test.parquet"))
y_train = pd.read_parquet(os.path.join(OUTPUTS_PATH, "y_train.parquet")).squeeze()
y_val   = pd.read_parquet(os.path.join(OUTPUTS_PATH, "y_val.parquet")).squeeze()
y_test  = pd.read_parquet(os.path.join(OUTPUTS_PATH, "y_test.parquet")).squeeze()

# Safety: ensure numeric, fill any residual NaN
X_train = X_train.select_dtypes(include=[np.number]).fillna(0)
X_val   = X_val[X_train.columns].fillna(0)
X_test  = X_test[X_train.columns].fillna(0)

print(f"X_train: {X_train.shape}  |  y_train success rate: {y_train.mean():.3f}")
print(f"X_val  : {X_val.shape}    |  y_val   success rate: {y_val.mean():.3f}")
print(f"X_test : {X_test.shape}   |  y_test  success rate: {y_test.mean():.3f}")

# ── staff_pick guard ──
if "staff_pick" in X_train.columns:
    print("\n⚠️  WARNING: staff_pick found in features — removing now (should have been removed in 03a)")
    X_train = X_train.drop(columns=["staff_pick"])
    X_val   = X_val.drop(columns=["staff_pick"])
    X_test  = X_test.drop(columns=["staff_pick"])
else:
    print("\n✅ staff_pick not present — correctly excluded in feature engineering (see 03a)")

# ── Apply QUICK_TEST ──
if QUICK_TEST:
    print(f"\n⚡ QUICK TEST MODE: using {QUICK_TEST_ROWS:,} train rows")
    X_train = X_train.iloc[:QUICK_TEST_ROWS]
    y_train = y_train.iloc[:QUICK_TEST_ROWS]
    X_val   = X_val.iloc[:QUICK_TEST_ROWS // 3]
    y_val   = y_val.iloc[:QUICK_TEST_ROWS // 3]
    N_SPLITS = 2
    N_ITER   = 5
else:
    print(f"\n🚀 FULL RUN MODE: using all {len(X_train):,} training rows")
    N_SPLITS = 5
    N_ITER   = 50

print(f"N_SPLITS={N_SPLITS}, N_ITER={N_ITER}")

## KNN — Excluded from v2 Pipeline

KNN removed from v2 pipeline. With 193 features, distance-based methods suffer from the curse of dimensionality — distances become meaningless in high-dimensional sparse spaces. Additionally, KNN's O(n) prediction time on 120K+ rows makes it impractical. In v1 it already required a 30K subsample and ranked last.

## Cross-Validation Setup

We use **TimeSeriesSplit** instead of StratifiedKFold because this is temporal data. Random CV would leak future data into training folds.

In [ ]:
tscv = TimeSeriesSplit(n_splits=N_SPLITS)
print(f"TimeSeriesSplit: n_splits={N_SPLITS}")
print(f"RandomizedSearchCV: n_iter={N_ITER}")

## Evaluation Helper

In [ ]:
results = []

def evaluate_and_record(name, model, X_tr, y_tr, X_te, y_te):
    """Fit on X_tr/y_tr, evaluate on X_te/y_te, append metrics to results list."""
    model.fit(X_tr, y_tr)

    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_te)[:, 1]
    elif hasattr(model, "decision_function"):
        raw = model.decision_function(X_te)
        y_prob = (raw - raw.min()) / (raw.max() - raw.min() + 1e-9)
    else:
        y_prob = model.predict(X_te).astype(float)

    y_pred = (y_prob >= 0.5).astype(int)

    auc  = roc_auc_score(y_te, y_prob) if len(np.unique(y_te)) > 1 else np.nan
    f1   = f1_score(y_te, y_pred, zero_division=0)
    prec = precision_score(y_te, y_pred, zero_division=0)
    rec  = recall_score(y_te, y_pred, zero_division=0)
    try:
        pr_auc = average_precision_score(y_te, y_prob)
    except Exception:
        pr_auc = np.nan

    row = {"model": name, "roc_auc": auc, "f1_binary": f1,
           "precision": prec, "recall": rec, "pr_auc": pr_auc}
    results.append(row)
    print(f"  {name:<40} AUC={auc:.4f}  F1={f1:.4f}  Prec={prec:.4f}  Rec={rec:.4f}")
    return model, y_prob

## Model 1 — Logistic Regression (Baseline)

Linear model; establishes the minimum acceptable benchmark. Class weights balanced to handle the ~62/38 imbalance.

In [ ]:
print("\n=== Model 1: Logistic Regression ===")
lr_search = RandomizedSearchCV(
    LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42),
    param_distributions={"C": [0.01, 0.1, 1.0, 10.0]},
    n_iter=min(N_ITER, 4),
    cv=tscv, scoring="roc_auc", n_jobs=-1, random_state=42
)
lr_search.fit(X_train, y_train)
lr_model, lr_proba = evaluate_and_record(
    "Logistic Regression", lr_search.best_estimator_,
    X_train, y_train, X_test, y_test)

## Model 2 — Decision Tree

Interpretable non-linear model; useful for understanding decision boundaries but prone to overfitting without depth constraints.

In [ ]:
print("\n=== Model 2: Decision Tree ===")
dt_search = RandomizedSearchCV(
    DecisionTreeClassifier(class_weight="balanced", random_state=42),
    param_distributions={
        "max_depth": [3, 5, 8, 12, None],
        "min_samples_leaf": [1, 5, 10, 20]
    },
    n_iter=min(N_ITER, 8),
    cv=tscv, scoring="roc_auc", n_jobs=-1, random_state=42
)
dt_search.fit(X_train, y_train)
dt_model, dt_proba = evaluate_and_record(
    "Decision Tree", dt_search.best_estimator_,
    X_train, y_train, X_test, y_test)

## Model 3 — Random Forest

`max_features=0.6` limits each split to 60% of features — important with 193 features to reduce correlation between trees and overfitting.

In [ ]:
print("\n=== Model 3: Random Forest ===")
rf_search = RandomizedSearchCV(
    RandomForestClassifier(class_weight="balanced", random_state=42, n_jobs=-1),
    param_distributions={
        "n_estimators": [100, 200, 300],
        "max_features": [0.3, 0.5, 0.6],
        "max_depth": [10, 20, None],
        "min_samples_leaf": [1, 2, 5]
    },
    n_iter=N_ITER,
    cv=tscv, scoring="roc_auc", n_jobs=-1, random_state=42
)
rf_search.fit(X_train, y_train)
rf_model, rf_proba = evaluate_and_record(
    "Random Forest", rf_search.best_estimator_,
    X_train, y_train, X_test, y_test)

# Save RF feature importances for notebook 05
rf_feat_imp = pd.Series(rf_model.feature_importances_, index=X_train.columns)
rf_feat_imp.sort_values(ascending=False).to_frame("importance").to_csv(
    os.path.join(RESULTS_PATH, "rf_importances.csv"))
print("✓ rf_importances.csv saved")

## Model 4 — Gradient Boosting

scikit-learn's native boosting implementation; slower than XGBoost but a useful cross-check.

In [ ]:
print("\n=== Model 4: Gradient Boosting ===")
gb_search = RandomizedSearchCV(
    GradientBoostingClassifier(random_state=42),
    param_distributions={
        "n_estimators": [100, 200],
        "learning_rate": [0.05, 0.1, 0.2],
        "max_depth": [3, 5],
        "subsample": [0.7, 0.9, 1.0]
    },
    n_iter=N_ITER,
    cv=tscv, scoring="roc_auc", n_jobs=-1, random_state=42
)
gb_search.fit(X_train, y_train)
gb_model, gb_proba = evaluate_and_record(
    "Gradient Boosting", gb_search.best_estimator_,
    X_train, y_train, X_test, y_test)

## Model 5 — XGBoost (Primary Model)

`colsample_bytree=0.6` is essential here: with 193 features (including 157 sparse binary TF-IDF columns), sampling 60% of features per tree reduces variance and prevents any single feature from dominating early splits. `scale_pos_weight` corrects for class imbalance.

In [ ]:
print("\n=== Model 5: XGBoost ===")
scale_pos = (y_train == 0).sum() / (y_train == 1).sum()

xgb_search = RandomizedSearchCV(
    XGBClassifier(
        scale_pos_weight=scale_pos,
        colsample_bytree=0.6,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    ),
    param_distributions={
        "n_estimators": [100, 200, 300],
        "learning_rate": [0.05, 0.1, 0.15],
        "max_depth": [4, 6, 8],
        "subsample": [0.7, 0.9],
        "reg_alpha": [0, 0.1, 1.0],
        "reg_lambda": [1.0, 2.0]
    },
    n_iter=N_ITER,
    cv=tscv, scoring="roc_auc", n_jobs=-1, random_state=42
)
xgb_search.fit(X_train, y_train)
xgb_model, xgb_proba = evaluate_and_record(
    "XGBoost", xgb_search.best_estimator_,
    X_train, y_train, X_test, y_test)

best_xgb = xgb_search.best_estimator_
print(f"\nBest XGB params: {xgb_search.best_params_}")

## Model 6 — Linear SVC (Calibrated)

SVMs with a linear kernel can work well in high-dimensional spaces. `CalibratedClassifierCV` wraps it to produce probability estimates (needed for ROC-AUC).

In [ ]:
print("\n=== Model 6: Linear SVC (Calibrated) ===")
svc_base = LinearSVC(class_weight="balanced", max_iter=2000, random_state=42, dual="auto")
svc_cal  = CalibratedClassifierCV(svc_base, cv=tscv)
svc_model, svc_proba = evaluate_and_record(
    "Linear SVC (Calibrated)", svc_cal,
    X_train, y_train, X_test, y_test)

## Model 7 — Naive Bayes

Strongest assumption (feature independence) but very fast and surprisingly robust in high-dimensional spaces when the goal is ranking rather than calibration.

In [ ]:
print("\n=== Model 7: Naive Bayes ===")
nb_model, nb_proba = evaluate_and_record(
    "Naive Bayes", GaussianNB(),
    X_train, y_train, X_test, y_test)

## Model 8 — Neural Network (MLP)

Multi-layer perceptron with 2–3 hidden layers. `early_stopping=True` prevents overfitting by monitoring a validation fraction of the training data. Random search over architecture and regularisation strength.

In [ ]:
print("\n=== Model 8: Neural Network (MLP) ===")
mlp_search = RandomizedSearchCV(
    MLPClassifier(
        random_state=42,
        max_iter=200,
        early_stopping=True,
        validation_fraction=0.1
    ),
    param_distributions={
        "hidden_layer_sizes": [(128, 64), (128, 64, 32), (256, 128), (64, 32)],
        "alpha": [0.0001, 0.001, 0.01],
        "learning_rate_init": [0.001, 0.01]
    },
    n_iter=min(N_ITER, 4),
    cv=tscv, scoring="roc_auc", random_state=42
)
mlp_search.fit(X_train, y_train)
mlp_model, mlp_proba = evaluate_and_record(
    "Neural Network (MLP)", mlp_search.best_estimator_,
    X_train, y_train, X_test, y_test)

## Model 9 — Voting Ensemble (LR + RF + XGB)

Soft-voting averages predicted probabilities from three diverse model families: a linear model, a bagging ensemble, and a boosting model. Diversity reduces correlated errors.

In [ ]:
print("\n=== Model 9: Voting Ensemble ===")
voting = VotingClassifier(
    estimators=[
        ("lr",  lr_search.best_estimator_),
        ("rf",  rf_search.best_estimator_),
        ("xgb", xgb_search.best_estimator_)
    ],
    voting="soft"
)
voting_model, voting_proba = evaluate_and_record(
    "Voting Ensemble (LR+RF+XGB)", voting,
    X_train, y_train, X_test, y_test)

## Model 10 — Stacking Ensemble

Base learners (LR, DT, RF, GB) produce out-of-fold predictions via TimeSeriesSplit; a Logistic Regression meta-learner is trained on those stacked predictions. Stacking can capture non-obvious combinations of base model strengths.

In [ ]:
print("\n=== Model 10: Stacking Ensemble ===")
stacking = StackingClassifier(
    estimators=[
        ("lr", lr_search.best_estimator_),
        ("dt", dt_search.best_estimator_),
        ("rf", rf_search.best_estimator_),
        ("gb", gb_search.best_estimator_)
    ],
    final_estimator=LogisticRegression(C=1.0, max_iter=1000, random_state=42),
    cv=tscv,
    passthrough=False
)
stacking_model, stacking_proba = evaluate_and_record(
    "Stacking Ensemble", stacking,
    X_train, y_train, X_test, y_test)

## Results — All Models Ranked by ROC-AUC

All metrics are on the **held-out test set only** (newest 20% of campaigns by launch date).

In [ ]:
results_df = (
    pd.DataFrame(results)
      .sort_values("roc_auc", ascending=False)
      .reset_index(drop=True)
)
results_df.index = results_df.index + 1

print("=" * 80)
print("MODEL RANKING (by ROC-AUC on test set)")
print("=" * 80)
print(results_df.to_string())

results_df.to_csv(os.path.join(RESULTS_PATH, "all_model_results.csv"), index=False)
print(f"\n✓ Results saved to {RESULTS_PATH}/all_model_results.csv")

best_name = results_df.iloc[0]["model"]
best_auc  = results_df.iloc[0]["roc_auc"]
lr_auc    = results_df.loc[results_df["model"] == "Logistic Regression", "roc_auc"].values[0]
print(f"\nBest model : {best_name} (AUC={best_auc:.4f})")
print(f"LR baseline: AUC={lr_auc:.4f}  |  Improvement: +{best_auc - lr_auc:.4f}")

## ROC Curves — All Models

In [ ]:
models_for_roc = [
    ("Logistic Regression",          lr_model,      lr_proba),
    ("Random Forest",                rf_model,      rf_proba),
    ("Gradient Boosting",            gb_model,      gb_proba),
    ("XGBoost",                      xgb_model,     xgb_proba),
    ("Neural Network (MLP)",         mlp_model,     mlp_proba),
    ("Voting Ensemble (LR+RF+XGB)",  voting_model,  voting_proba),
]

fig, ax = plt.subplots(figsize=(10, 7))
for name, model, proba in models_for_roc:
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    ax.plot(fpr, tpr, lw=2, label=f"{name} (AUC={auc:.3f})")

ax.plot([0,1],[0,1],'k--', lw=1, label="Random")
ax.set_xlabel("False Positive Rate", fontsize=11)
ax.set_ylabel("True Positive Rate", fontsize=11)
ax.set_title("ROC Curves — All Models (Test Set)", fontweight="bold", fontsize=12)
ax.legend(loc="lower right", fontsize=9)
ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_PATH, "roc_curves.png"), dpi=150, bbox_inches="tight")
plt.show()
print("✓ ROC curves saved")

## Feature Importance — XGBoost (Top 20)

XGBoost's built-in importance scores (gain) show which features contribute most to split quality across all trees.

In [ ]:
feat_imp = pd.Series(best_xgb.feature_importances_, index=X_train.columns)
top20 = feat_imp.nlargest(20)

fig, ax = plt.subplots(figsize=(10, 7))
top20.sort_values().plot.barh(ax=ax, color="#3B82F6", edgecolor="white")
ax.set_title("Top 20 Feature Importances (XGBoost — Gain)", fontweight="bold")
ax.set_xlabel("Importance Score")
ax.grid(axis="x", alpha=0.2)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_PATH, "feature_importance_xgb.png"), dpi=150, bbox_inches="tight")
plt.show()
print("✓ Feature importance plot saved")

# Also save to CSV for notebook 05
feat_imp.sort_values(ascending=False).to_frame("importance").to_csv(
    os.path.join(RESULTS_PATH, "xgb_importances.csv"))
print("✓ xgb_importances.csv saved")

## SHAP Analysis — XGBoost (Best Model)

**SHAP (SHapley Additive exPlanations)** provides theoretically grounded feature attributions. Unlike raw feature importance (which measures how often a feature is used), SHAP values show *how much* each feature *pushed the prediction up or down* for each individual campaign.

The summary plot shows:
- **Feature order** (y-axis): ranked by mean |SHAP| across all samples — most impactful at top
- **Point colour**: red = high feature value, blue = low feature value
- **Point x-position**: positive = pushed prediction toward success, negative = pushed toward failure

This lets us confirm directionality: does a higher `log_goal` hurt or help? Does having a video help? SHAP makes the model's reasoning transparent.

In [ ]:
if HAS_SHAP:
    print("Computing SHAP values...")
    shap_n = 500 if QUICK_TEST else 1000
    X_shap = X_test.iloc[:shap_n]

    explainer  = shap.TreeExplainer(best_xgb)
    shap_vals  = explainer.shap_values(X_shap)

    plt.figure(figsize=(12, 8))
    shap.summary_plot(shap_vals, X_shap,
                      feature_names=X_shap.columns.tolist(),
                      max_display=20, show=False)
    plt.title("SHAP Summary Plot — XGBoost (Top 20 Features)", fontweight="bold")
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_PATH, "shap_summary.png"), dpi=150, bbox_inches="tight")
    plt.show()
    print("✓ SHAP summary plot saved")

    # Save mean |SHAP| as feature importances
    mean_abs_shap = pd.Series(
        np.abs(shap_vals).mean(axis=0),
        index=X_shap.columns
    ).sort_values(ascending=False)
    mean_abs_shap.to_frame("mean_abs_shap").to_csv(
        os.path.join(RESULTS_PATH, "shap_importances.csv"))
    print("✓ shap_importances.csv saved")
else:
    print("[SKIP] SHAP not installed. Run: pip install shap  then re-run this cell.")

## Confusion Matrix — Best XGBoost Model

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_test, (xgb_proba >= 0.5).astype(int))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Failed", "Successful"])
disp.plot(ax=ax, colorbar=False, cmap="Blues")
ax.set_title("Confusion Matrix — XGBoost (threshold = 0.5)", fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_PATH, "confusion_matrix_xgb.png"), dpi=150, bbox_inches="tight")
plt.show()
print("✓ Confusion matrix saved")

tn, fp, fn, tp = cm.ravel()
print(f"\nTrue Negatives  (correctly predicted failures) : {tn:,}")
print(f"False Positives (failures predicted as success) : {fp:,}")
print(f"False Negatives (successes predicted as failure): {fn:,}")
print(f"True Positives  (correctly predicted successes) : {tp:,}")
print(f"\nPrecision: {tp/(tp+fp):.3f}  |  Recall: {tp/(tp+fn):.3f}")

## Validation Set Check — XGBoost

We compare XGBoost performance on val vs test to check for overfitting or distributional drift between the two held-out periods.

In [ ]:
xgb_val_prob = best_xgb.predict_proba(X_val)[:, 1]
xgb_val_auc  = roc_auc_score(y_val, xgb_val_prob)
xgb_test_auc = roc_auc_score(y_test, xgb_proba)

print(f"XGBoost — Validation AUC : {xgb_val_auc:.4f}")
print(f"XGBoost — Test AUC       : {xgb_test_auc:.4f}")
print(f"Val → Test delta         : {xgb_test_auc - xgb_val_auc:+.4f}")

if abs(xgb_val_auc - xgb_test_auc) > 0.05:
    print("\n⚠️  Large val/test gap — possible distributional drift or overfitting")
else:
    print("\n✅ Val/test gap is small — model generalises well to the test period")

## Summary & Next Steps

In [ ]:
print("=" * 80)
print("MODELLING COMPLETE")
print("=" * 80)
print(f"\nBest model : {results_df.iloc[0]['model']}")
print(f"Test AUC   : {results_df.iloc[0]['roc_auc']:.4f}")
print(f"Test F1    : {results_df.iloc[0]['f1_binary']:.4f}")
print(f"\nAll model results → {RESULTS_PATH}/all_model_results.csv")
print(f"Figures saved     → {FIGURES_PATH}/")
print(f"\n→ Proceed to 05_reflection.ipynb for critical analysis and business insights")